In [21]:
import pandas as pd
import numpy as np
from sklearn.neural_network import MLPClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.model_selection import GroupKFold
from sklearn.metrics import classification_report
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import joblib
from sklearn.utils import compute_class_weight

# 1. Data Loading and Preprocessing
def load_and_preprocess():
    data = pd.read_csv('full_data_unhealthy_imputed.csv')
    
    # Feature engineering
    data['date'] = pd.to_datetime(data['date'])
    data['rest_to_eat_ratio'] = np.where(
        (data['EAT'] == 0) & (data['REST'] == 0),
        1.0,
        data['REST'] / (data['EAT'] + 1e-6)
    )
    
    # Cyclical time features
    data['hour_sin'] = np.sin(2 * np.pi * data['hour'] / 24)
    data['hour_cos'] = np.cos(2 * np.pi * data['hour'] / 24)
    
    features = ['IN_ALLEYS', 'REST', 'EAT', 'ACTIVITY_LEVEL', 
               'rest_to_eat_ratio', 'hour_sin', 'hour_cos']
    targets = ['oestrus', 'calving', 'lameness', 'mastitis']
    
    return data[features], data[targets], data['cow']

# 2. Custom Class Weight Calculation
def calculate_class_weights(y):
    class_weights = []
    for i, target in enumerate(y.columns):
        classes = np.unique(y.iloc[:,i])
        weights = compute_class_weight('balanced', classes=classes, y=y.iloc[:,i])
        class_weights.append(dict(zip(classes, weights)))
    return class_weights

# 3. MLP Model Definition with Class Weights
def create_mlp_model():
    # Create base MLP model
    base_mlp = MLPClassifier(
        hidden_layer_sizes=(128, 64),
        activation='relu',
        solver='adam',
        alpha=0.001,
        batch_size=512,
        learning_rate='adaptive',
        early_stopping=True,
        max_iter=1000,
        random_state=42
    )
    
    # Create pipeline
    model = Pipeline([
        ('scaler', StandardScaler()),
        ('mlp', MultiOutputClassifier(base_mlp))
    ])
    
    return model

# 4. Training with GroupKFold and Class Weights
def train_mlp_model():
    X, y, groups = load_and_preprocess()
    class_weights = calculate_class_weights(y)
    model = create_mlp_model()
    
    gkf = GroupKFold(n_splits=3)
    for fold, (train_idx, test_idx) in enumerate(gkf.split(X, y, groups)):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
        
        print(f"\nTraining Fold {fold+1}...")
        
        # Apply class weights by modifying the input data
        # Duplicate minority class samples to balance classes
        max_count = max(y_train.sum(axis=0))
        resampled_X = []
        resampled_y = []
        
        for i, target in enumerate(y.columns):
            minority_indices = y_train[y_train[target] == 1].index
            repeat_times = int(max_count / len(minority_indices)) - 1
            if repeat_times > 0:
                resampled_X.append(pd.concat([X_train.loc[minority_indices]] * repeat_times))
                resampled_y.append(pd.concat([y_train.loc[minority_indices]] * repeat_times))
        
        if resampled_X:
            X_train = pd.concat([X_train] + resampled_X)
            y_train = pd.concat([y_train] + resampled_y)
        
        model.fit(X_train, y_train)
        
        # Predict with adjusted thresholds
        y_proba = model.predict_proba(X_test)
        thresholds = [0.3, 0.2, 0.4, 0.1]  # Disease-specific thresholds
        y_pred = np.array([(y_proba[i][:,1] > thresholds[i]).astype(int) for i in range(4)]).T
        
        for i, target in enumerate(y.columns):
            print(f"\n{target} Classification Report:")
            print(classification_report(y_test.iloc[:,i], y_pred[:,i]))
    
    joblib.dump(model, 'mlp_cattle_model.joblib')
    print("\nMLP model training complete and saved.")

# 5. Prediction Function
def predict_with_mlp(new_data):
    model = joblib.load('mlp_cattle_model.joblib')
    
    # Feature engineering
    new_data = new_data.copy()
    new_data['rest_to_eat_ratio'] = np.where(
        (new_data['EAT'] == 0) & (new_data['REST'] == 0),
        1.0,
        new_data['REST'] / (new_data['EAT'] + 1e-6)
    )
    new_data['hour_sin'] = np.sin(2 * np.pi * new_data['hour'] / 24)
    new_data['hour_cos'] = np.cos(2 * np.pi * new_data['hour'] / 24)
    
    features = ['IN_ALLEYS', 'REST', 'EAT', 'ACTIVITY_LEVEL', 
               'rest_to_eat_ratio', 'hour_sin', 'hour_cos']
    X_new = new_data[features]
    
    # Get probabilities
    y_proba = model.predict_proba(X_new)
    thresholds = [0.3, 0.2, 0.4, 0.1]
    results = pd.DataFrame({
        'oestrus': (y_proba[0][:,1] > thresholds[0]).astype(int),
        'oestrus_prob': y_proba[0][:,1],
        'calving': (y_proba[1][:,1] > thresholds[1]).astype(int),
        'calving_prob': y_proba[1][:,1],
        'lameness': (y_proba[2][:,1] > thresholds[2]).astype(int),
        'lameness_prob': y_proba[2][:,1],
        'mastitis': (y_proba[3][:,1] > thresholds[3]).astype(int),
        'mastitis_prob': y_proba[3][:,1]
    })
    
    return results

# Example usage
if __name__ == "__main__":
    # Train the model
    train_mlp_model()
    
    # Example prediction
    new_data = pd.DataFrame({
        'IN_ALLEYS': [35.6, 0.0, 10.2],
        'REST': [3564.4, 3599.9, 2500.0],
        'EAT': [0.0, 0.0, 500.0],
        'ACTIVITY_LEVEL': [-814.1, -827.9, -700.0],
        'hour': [1, 2, 3],
        'date': ['2023-01-01', '2023-01-02', '2023-01-03']
    })
    
    predictions = predict_with_mlp(new_data)
    print("\nMLP Predictions:")
    print(predictions)


Training Fold 1...

oestrus Classification Report:


C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


              precision    recall  f1-score   support

           0       1.00      1.00      1.00    642649
           1       0.00      0.00      0.00      2544

    accuracy                           1.00    645193
   macro avg       0.50      0.50      0.50    645193
weighted avg       0.99      1.00      0.99    645193


calving Classification Report:


C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


              precision    recall  f1-score   support

           0       1.00      1.00      1.00    643777
           1       0.00      0.00      0.00      1416

    accuracy                           1.00    645193
   macro avg       0.50      0.50      0.50    645193
weighted avg       1.00      1.00      1.00    645193


lameness Classification Report:


...

              precision    recall  f1-score   support

           0       1.00      1.00      1.00    643652
           1       0.00      0.00      0.00      1512

    accuracy                           1.00    645164
   macro avg       0.50      0.50      0.50    645164
weighted avg       1.00      1.00      1.00    645164


lameness Classification Report:


C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


              precision    recall  f1-score   support

           0       1.00      1.00      1.00    644228
           1       0.00      0.00      0.00       936

    accuracy                           1.00    645164
   macro avg       0.50      0.50      0.50    645164
weighted avg       1.00      1.00      1.00    645164


mastitis Classification Report:


C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


              precision    recall  f1-score   support

           0       1.00      1.00      1.00    644636
           1       0.00      0.00      0.00       528

    accuracy                           1.00    645164
   macro avg       0.50      0.50      0.50    645164
weighted avg       1.00      1.00      1.00    645164


MLP model training complete and saved.

MLP Predictions:
   oestrus  oestrus_prob  calving  calving_prob  lameness  lameness_prob  \
0        0      0.002227        0      0.000359         0       0.002711   
1        0      0.001915        0      0.000331         0       0.002603   
2        0      0.002906        0      0.000977         0       0.002645   

   mastitis  mastitis_prob  
0         0       0.003800  
1         0       0.003848  
2         0       0.003291  


In [27]:
import pandas as pd
import numpy as np
from sklearn.neural_network import MLPClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.model_selection import GroupKFold
from sklearn.metrics import classification_report
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import joblib
from sklearn.utils import compute_class_weight

# 1. Data Loading and Preprocessing
def load_and_preprocess():
    data = pd.read_csv('full_data_unhealthy_imputed.csv')
    
    # Feature engineering
    data['date'] = pd.to_datetime(data['date'])
    data['rest_to_eat_ratio'] = np.where(
        (data['EAT'] == 0) & (data['REST'] == 0),
        1.0,
        data['REST'] / (data['EAT'] + 1e-6)
    )
    
    # Cyclical time features
    data['hour_sin'] = np.sin(2 * np.pi * data['hour'] / 24)
    data['hour_cos'] = np.cos(2 * np.pi * data['hour'] / 24)
    
    features = ['IN_ALLEYS', 'REST', 'EAT', 'ACTIVITY_LEVEL', 
               'rest_to_eat_ratio', 'hour_sin', 'hour_cos']
    targets = ['oestrus', 'calving', 'lameness', 'mastitis']
    
    return data[features], data[targets], data['cow']

# 2. Custom Class Weight Calculation
def calculate_class_weights(y):
    class_weights = []
    for i, target in enumerate(y.columns):
        classes = np.unique(y.iloc[:,i])
        weights = compute_class_weight('balanced', classes=classes, y=y.iloc[:,i])
        class_weights.append(dict(zip(classes, weights)))
    return class_weights

# 3. MLP Model Definition with Class Weights
def create_mlp_model(class_weights):
    # Create base MLP model with parameters
    base_mlp = MLPClassifier(
        hidden_layer_sizes=(128, 64),
        activation='relu',
        solver='adam',
        alpha=0.001,
        batch_size=512,
        learning_rate='adaptive',
        early_stopping=True,
        max_iter=1000,
        random_state=42
    )
    
    # Create MultiOutputClassifier
    multi_output = MultiOutputClassifier(base_mlp)
    
    # Create simple pipeline with just scaler and classifier
    model = Pipeline([
        ('scaler', StandardScaler()),
        ('classifier', multi_output)
    ])
    
    return model

# 4. Training with GroupKFold
def train_mlp_model():
    X, y, groups = load_and_preprocess()
    class_weights = calculate_class_weights(y)
    model = create_mlp_model(class_weights)
    
    gkf = GroupKFold(n_splits=3)
    for fold, (train_idx, test_idx) in enumerate(gkf.split(X, y, groups)):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
        
        print(f"\nTraining Fold {fold+1}...")
        
        # Apply class weights by modifying the input data
        # We'll duplicate minority class samples to balance classes
        max_count = max(y_train.sum(axis=0))
        resampled_X = []
        resampled_y = []
        
        for i, target in enumerate(y.columns):
            minority_indices = y_train[y_train[target] == 1].index
            repeat_times = int(max_count / len(minority_indices)) - 1
            if repeat_times > 0:
                resampled_X.append(pd.concat([X_train.loc[minority_indices]] * repeat_times))
                resampled_y.append(pd.concat([y_train.loc[minority_indices]] * repeat_times))
        
        if resampled_X:
            X_train = pd.concat([X_train] + resampled_X)
            y_train = pd.concat([y_train] + resampled_y)
        
        model.fit(X_train, y_train)
        
        # Predict with adjusted thresholds
        y_proba = model.predict_proba(X_test)
        thresholds = [0.3, 0.2, 0.4, 0.1]  # Disease-specific thresholds
        y_pred = np.array([(y_proba[i][:,1] > thresholds[i]).astype(int) for i in range(4)]).T
        
        for i, target in enumerate(y.columns):
            print(f"\n{target} Classification Report (Threshold={thresholds[i]}):")
            print(classification_report(y_test.iloc[:,i], y_pred[:,i], zero_division=0))
    
    joblib.dump(model, 'mlp_cattle_model.joblib')
    print("\nMLP model training complete and saved.")

# 5. Prediction Function
def predict_with_mlp(new_data):
    model = joblib.load('mlp_cattle_model.joblib')
    
    # Feature engineering
    new_data = new_data.copy()
    new_data['rest_to_eat_ratio'] = np.where(
        (new_data['EAT'] == 0) & (new_data['REST'] == 0),
        1.0,
        new_data['REST'] / (new_data['EAT'] + 1e-6)
    )
    new_data['hour_sin'] = np.sin(2 * np.pi * new_data['hour'] / 24)
    new_data['hour_cos'] = np.cos(2 * np.pi * new_data['hour'] / 24)
    
    features = ['IN_ALLEYS', 'REST', 'EAT', 'ACTIVITY_LEVEL', 
               'rest_to_eat_ratio', 'hour_sin', 'hour_cos']
    X_new = new_data[features]
    
    # Get probabilities
    y_proba = model.predict_proba(X_new)
    thresholds = [0.3, 0.2, 0.4, 0.1]
    results = pd.DataFrame({
        'oestrus': (y_proba[0][:,1] > thresholds[0]).astype(int),
        'oestrus_prob': y_proba[0][:,1],
        'calving': (y_proba[1][:,1] > thresholds[1]).astype(int),
        'calving_prob': y_proba[1][:,1],
        'lameness': (y_proba[2][:,1] > thresholds[2]).astype(int),
        'lameness_prob': y_proba[2][:,1],
        'mastitis': (y_proba[3][:,1] > thresholds[3]).astype(int),
        'mastitis_prob': y_proba[3][:,1]
    })
    
    return results

# Example usage
if __name__ == "__main__":
    # Train the model
    train_mlp_model()
    
    # Example prediction
    new_data = pd.DataFrame({
        'IN_ALLEYS': [35.6, 0.0, 10.2],
        'REST': [3564.4, 3599.9, 2500.0],
        'EAT': [0.0, 0.0, 500.0],
        'ACTIVITY_LEVEL': [-814.1, -827.9, -700.0],
        'hour': [1, 2, 3],
        'date': ['2023-01-01', '2023-01-02', '2023-01-03']
    })
    
    predictions = predict_with_mlp(new_data)
    print("\nMLP Predictions:")
    print(predictions)


Training Fold 1...

oestrus Classification Report (Threshold=0.3):
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    642649
           1       0.00      0.00      0.00      2544

    accuracy                           1.00    645193
   macro avg       0.50      0.50      0.50    645193
weighted avg       0.99      1.00      0.99    645193


calving Classification Report (Threshold=0.2):
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    643777
           1       0.00      0.00      0.00      1416

    accuracy                           1.00    645193
   macro avg       0.50      0.50      0.50    645193
weighted avg       1.00      1.00      1.00    645193


lameness Classification Report (Threshold=0.4):
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    644089
           1       0.00      0.00      0.00      1104

    accuracy     